# Vector Databases with LangChain — FAISS, Chroma, Qdrant, Milvus (all local)

`l21_rag_pipeline_from_scratch.ipynb` built retrieval by hand: a plain Python list of vectors
and a from-scratch cosine-similarity loop, deliberately with no vector database. This notebook
is the natural next step — the same job (store vectors, find the nearest ones to a query), done
by four real vector databases through LangChain's common `VectorStore` interface:

1. **FAISS** — a similarity-search *library* (not a client-server database), in-process.
2. **Chroma** — an embedded vector database, single local file/folder.
3. **Qdrant** — normally a client-server database; used here in its local, serverless mode.
4. **Milvus** — normally a distributed cluster; used here via **Milvus Lite**, its embedded mode.

**Everything here runs fully local** — no cloud vector-DB account, no Docker, no server process
for any of the four. The only network call in this notebook is the embeddings API (OpenAI, same
as every other notebook in this course) — that's a separate, already-accepted dependency, not a
vector-DB-as-a-service dependency.

Each section also ends with a short **scaling** note — informational only, not something we
spin up here — on how that same store looks in a real production deployment.


In [1]:
import shutil
from pathlib import Path

from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_pdfmuse import PdfmuseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

docs = PdfmuseLoader("data/sampled_schemes.pdf").load()
chunks = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=0).split_documents(docs)
print(len(chunks), "chunks from", len(docs), "pages")
VECTOR_STORES_DIR= Path("vector_stores")

QUERY = "What subsidies are available for farm machinery and equipment?"



shutil.rmtree(VECTOR_STORES_DIR, ignore_errors=True)  # clean slate so this notebook re-runs cleanly
VECTOR_STORES_DIR.mkdir(exist_ok=True)


c:\Users\hp\Desktop\genai python\genai_class\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


88 chunks from 9 pages


## 1. FAISS

FAISS (Facebook AI Similarity Search) is a C++/Python *library* for nearest-neighbor search —
there's no server, no client-server protocol, just an in-process index. LangChain's integration
still lives in `langchain-community` (that package as a whole carries a sunset notice, but
nothing FAISS-specific has moved to a dedicated package, so this remains the current import).


In [2]:
from langchain_community.vectorstores import FAISS

faiss_path = str(VECTOR_STORES_DIR / "faiss_index")

faiss_store = FAISS.from_documents(chunks, embeddings)
faiss_store.save_local(faiss_path)

# Reload from disk to prove persistence actually round-trips, not just the in-memory object
faiss_store = FAISS.load_local(faiss_path, embeddings, allow_dangerous_deserialization=True)
print("FAISS index reloaded from", faiss_path)


C:\Users\hp\AppData\Local\Temp\ipykernel_12768\479070063.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


FAISS index reloaded from vector_stores\faiss_index


In [3]:
for doc, score in faiss_store.similarity_search_with_score(QUERY, k=3):
    print(f"[{score:.4f}] {doc.page_content[:150].replace(chr(10), ' ')}")


[0.8007] Interventions:  Provision  of  subsidy  to  farmers  on  the  selected  improved  farm  machineries & equipment’s used for various agricultural operat
[0.9209] Farm Machineries – Distribution of farm equipment and tools like conoweeder,  manual sprayer, drum seeder, seed drill, zero till multi­crop planter, p
[0.9280] Objectives: The prime objective would be to promote agriculture mechanization  through extending procurement subsidy & establishment of Farm Machinery


That score is an L2 *distance* — lower means more similar. This matters below: Chroma (left at
its default) and Milvus (explicitly configured with `metric_type="L2"`) use the same
lower-is-better convention as FAISS, while Qdrant is explicitly configured with cosine distance
below and returns a similarity-style score where *higher* is more similar — the direction isn't
standardized across stores, and LangChain doesn't normalize it for you. (Verified by actually
comparing the printed scores across all four stores later in this notebook, not assumed.)


In [4]:
faiss_retriever = faiss_store.as_retriever(search_type="similarity", search_kwargs={"k": 3})
faiss_retriever.invoke(QUERY)


[Document(id='c1c49257-851d-4e8a-9462-ec044dd5c717', metadata={'source': 'data/sampled_schemes.pdf', 'source_kind': 'Pdf', 'page': 2, 'total_pages': 9}, page_content='Interventions:\xa0 Provision\xa0 of\xa0 subsidy\xa0 to\xa0 farmers\xa0 on\xa0 the\xa0 selected\xa0 improved\xa0 farm\xa0 machineries\xa0&\xa0equipment’s\xa0used\xa0for\xa0various\xa0agricultural\xa0operations\xa0(both\xa0during\xa0 crop\xa0stand\xa0&\xa0post\xadharvest),\xa0conducting\xa0training\xa0&\xa0demonstrations\xa0on\xa0their\xa0use\xa0and\xa0 maintenance.\xa0 Besides,\xa0 assistance\xa0 would\xa0 be'),
 Document(id='0c7df6a6-9470-46eb-9d3a-4d39fec0e6cf', metadata={'source': 'data/sampled_schemes.pdf', 'source_kind': 'Pdf', 'page': 0, 'total_pages': 9}, page_content='Farm\xa0Machineries\xa0–\xa0Distribution\xa0of\xa0farm\xa0equipment\xa0and\xa0tools\xa0like\xa0conoweeder,\xa0 manual\xa0sprayer,\xa0drum\xa0seeder,\xa0seed\xa0drill,\xa0zero\xa0till multi\xadcrop\xa0planter,\xa0power\xa0 weeder,\xa0paddy\xa0thresher,

**Scaling, in brief:** FAISS is a library, not a deployable service — there's nothing to scale
*out*, only up. Everything happens in one process, so scaling means (a) fitting more vectors in
RAM on a single machine using compressed index types (IVF, PQ, HNSW variants) instead of the
brute-force flat index used above, or (b) sharding the dataset across multiple FAISS indexes /
processes / machines yourself and merging results in application code — FAISS gives you no
built-in clustering, replication, or multi-node query routing. It's an excellent fit when the
whole index reasonably fits in RAM (or on local disk via memory-mapping) on one box and you
already have — or don't need — infrastructure around it.


## 2. Chroma

Chroma is an embedded vector database — no server process, backed by a local file/folder. The
current `langchain-chroma` API auto-persists to `persist_directory`; older Chroma versions
required an explicit `.persist()` call, which no longer exists on the class. Left unconfigured
(as below), a new Chroma collection defaults to squared-L2 distance, not cosine — same
lower-is-better convention as FAISS above; passing
`collection_metadata={"hnsw:space": "cosine"}` would switch it.


In [6]:
from langchain_chroma import Chroma

chroma_store = Chroma(
    collection_name="hdfc_schemes",
    embedding_function=embeddings,
    persist_directory=str(VECTOR_STORES_DIR / "chroma_db"),
)
chroma_store.add_documents(chunks)
print("Chroma collection count:", chroma_store._collection.count())


Chroma collection count: 88


In [7]:
for doc, score in chroma_store.similarity_search_with_score(QUERY, k=3):
    print(f"[{score:.4f}] {doc.page_content[:150].replace(chr(10), ' ')}")


[0.8007] Interventions:  Provision  of  subsidy  to  farmers  on  the  selected  improved  farm  machineries & equipment’s used for various agricultural operat
[0.9209] Farm Machineries – Distribution of farm equipment and tools like conoweeder,  manual sprayer, drum seeder, seed drill, zero till multi­crop planter, p
[0.9280] Objectives: The prime objective would be to promote agriculture mechanization  through extending procurement subsidy & establishment of Farm Machinery


In [8]:
chroma_retriever = chroma_store.as_retriever(search_type="similarity", search_kwargs={"k": 3})
chroma_retriever.invoke(QUERY)


[Document(id='ba3af0a4-19eb-47d3-ac58-438fe6bf835e', metadata={'page': 2, 'total_pages': 9, 'source': 'data/sampled_schemes.pdf', 'source_kind': 'Pdf'}, page_content='Interventions:\xa0 Provision\xa0 of\xa0 subsidy\xa0 to\xa0 farmers\xa0 on\xa0 the\xa0 selected\xa0 improved\xa0 farm\xa0 machineries\xa0&\xa0equipment’s\xa0used\xa0for\xa0various\xa0agricultural\xa0operations\xa0(both\xa0during\xa0 crop\xa0stand\xa0&\xa0post\xadharvest),\xa0conducting\xa0training\xa0&\xa0demonstrations\xa0on\xa0their\xa0use\xa0and\xa0 maintenance.\xa0 Besides,\xa0 assistance\xa0 would\xa0 be'),
 Document(id='df4f1703-fed2-45bf-a575-0e118ab3e5e2', metadata={'page': 0, 'source': 'data/sampled_schemes.pdf', 'source_kind': 'Pdf', 'total_pages': 9}, page_content='Farm\xa0Machineries\xa0–\xa0Distribution\xa0of\xa0farm\xa0equipment\xa0and\xa0tools\xa0like\xa0conoweeder,\xa0 manual\xa0sprayer,\xa0drum\xa0seeder,\xa0seed\xa0drill,\xa0zero\xa0till multi\xadcrop\xa0planter,\xa0power\xa0 weeder,\xa0paddy\xa0thresher,

**Scaling, in brief:** the embedded mode used above is single-node by design — great for local
dev and moderate-sized collections (comfortably into the millions of vectors on one machine).
Chroma also has a client-server mode (`chromadb.HttpClient`, running as its own process) and a
hosted Chroma Cloud for scaling beyond one machine; the embedded mode here is the prototyping end
of that spectrum, not the horizontally-scaled end.


## 3. Qdrant

Qdrant is normally a client-server database (Rust, REST/gRPC). `qdrant-client` also has a local
mode with **no server process** — either fully in-memory (`QdrantClient(":memory:")`, LangChain's
own quickstart example) or persisted to a local folder (`QdrantClient(path=...)`, used below so
this section demonstrates on-disk persistence like the other three).


In [10]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore

qdrant_client = QdrantClient(path=str(VECTOR_STORES_DIR / "qdrant_db"))
vector_size = len(embeddings.embed_query("sample text"))

qdrant_client.create_collection(
    collection_name="schemes",
    vectors_config=VectorParams(size=vector_size, distance=Distance.COSINE),
)

qdrant_store = QdrantVectorStore(client=qdrant_client, collection_name="schemes", embedding=embeddings)
qdrant_store.add_documents(chunks)


['24a7cffccc3c4b8fa94411a924bc33b8',
 '75b3266bfe4f4dc1a0c31de3538f28df',
 '63158fbd182d4a7eaf30b04f96a23270',
 'e6eda78131a841eda29ef126d4b76bac',
 '0fb10274f2974187896c29ab9b65fb4a',
 'dc697c06e3114857baa967bb7686d5a8',
 '234211bab20940e686b4cba3840456a6',
 '1c885c67931e473eb99dcad3823b71eb',
 'f925d476c2464248ba75a8a1fd703af4',
 '7f31651ecbe54cc4b3ebe955bad055d8',
 'c9ef142952d04fa8ad45d8cdd1823eba',
 '103931812baf4ef3b1c6b769b3456f3e',
 '07b28eac5580413fa5fbc8e54421aaae',
 '85f1e9ed5c4445598abf3ff4efd2a655',
 'a4172ebb76aa490481f11236e3f0fb36',
 '4b2e380530964ceab80654dae7c229aa',
 'cca59541535d40698472a65f23a82bf5',
 '633466caee45402fa6040434f80f914d',
 'bf7144ca474b48b3abf092dcc199c4f4',
 '15414bbc9ad24340b72613b222d5065a',
 '7b246d1cd59244af80b6cad3c5cc14ca',
 '3dd53e5dc6ee47ad996aca8fd8f966cf',
 'fff2dd390f37468face96d04d05375b3',
 '56a5c71e8b144a45a1317781a64bf053',
 'a4a616140c2d47099a2b97a1356efd07',
 'b151ca8eb9874512b644158d2f15fb49',
 '91fdbf329ea74ca08a59ff8656da3f6e',
 

In [11]:
for doc, score in qdrant_store.similarity_search_with_score(QUERY, k=3):
    print(f"[{score:.4f}] {doc.page_content[:150].replace(chr(10), ' ')}")


[0.5998] Interventions:  Provision  of  subsidy  to  farmers  on  the  selected  improved  farm  machineries & equipment’s used for various agricultural operat
[0.5396] Farm Machineries – Distribution of farm equipment and tools like conoweeder,  manual sprayer, drum seeder, seed drill, zero till multi­crop planter, p
[0.5362] Objectives: The prime objective would be to promote agriculture mechanization  through extending procurement subsidy & establishment of Farm Machinery


In [12]:
qdrant_retriever = qdrant_store.as_retriever(search_type="similarity", search_kwargs={"k": 3})
qdrant_retriever.invoke(QUERY)


[Document(metadata={'source': 'data/sampled_schemes.pdf', 'source_kind': 'Pdf', 'page': 2, 'total_pages': 9, '_id': 'a4a616140c2d47099a2b97a1356efd07', '_collection_name': 'schemes'}, page_content='Interventions:\xa0 Provision\xa0 of\xa0 subsidy\xa0 to\xa0 farmers\xa0 on\xa0 the\xa0 selected\xa0 improved\xa0 farm\xa0 machineries\xa0&\xa0equipment’s\xa0used\xa0for\xa0various\xa0agricultural\xa0operations\xa0(both\xa0during\xa0 crop\xa0stand\xa0&\xa0post\xadharvest),\xa0conducting\xa0training\xa0&\xa0demonstrations\xa0on\xa0their\xa0use\xa0and\xa0 maintenance.\xa0 Besides,\xa0 assistance\xa0 would\xa0 be'),
 Document(metadata={'source': 'data/sampled_schemes.pdf', 'source_kind': 'Pdf', 'page': 0, 'total_pages': 9, '_id': '1c885c67931e473eb99dcad3823b71eb', '_collection_name': 'schemes'}, page_content='Farm\xa0Machineries\xa0–\xa0Distribution\xa0of\xa0farm\xa0equipment\xa0and\xa0tools\xa0like\xa0conoweeder,\xa0 manual\xa0sprayer,\xa0drum\xa0seeder,\xa0seed\xa0drill,\xa0zero\xa0till multi\

**Scaling, in brief:** production Qdrant runs as a server, or a distributed cluster with sharding
and replication across nodes, plus payload (metadata) filtering integrated directly into its HNSW
index. The important thing for this notebook: `QdrantVectorStore`'s API doesn't change between
the local embedded mode above and a full cluster — swapping `QdrantClient(path=...)` for
`QdrantClient(url="https://your-cluster:6333")` is a connection change, not a code rewrite.


## 4. Milvus

Milvus is normally a distributed, cloud-native cluster (separate proxy/query/data/index node
types deployed via Kubernetes, or Zilliz Cloud). **Milvus Lite** is its embedded mode — a single
local `.db` file, no server, no cluster — invoked simply by pointing `connection_args` at a local
file path instead of a server URL. `index_params` below explicitly requests L2 distance, same
convention as FAISS and (by default) Chroma.


In [ ]:
from langchain_milvus import Milvus

milvus_store = Milvus(
    embedding_function=embeddings,
    connection_args={"uri": str(VECTOR_STORES_DIR / "milvus_demo.db")},
    index_params={"index_type": "FLAT", "metric_type": "L2"},
)
milvus_store.add_documents(chunks)


c:\Users\hp\Desktop\genai python\genai_class\.venv\Lib\site-packages\langchain_milvus\vectorstores\milvus.py:1408: UserWarning: No ids provided and auto_id is False. Setting auto_id to True automatically.
  warnings.warn(


[1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49,
 50,
 51,
 52,
 53,
 54,
 55,
 56,
 57,
 58,
 59,
 60,
 61,
 62,
 63,
 64,
 65,
 66,
 67,
 68,
 69,
 70,
 71,
 72,
 73,
 74,
 75,
 76,
 77,
 78,
 79,
 80,
 81,
 82,
 83,
 84,
 85,
 86,
 87,
 88]

In [16]:
for doc, score in milvus_store.similarity_search_with_score(QUERY, k=3):
    print(f"[{score:.4f}] {doc.page_content[:150].replace(chr(10), ' ')}")


[0.8007] Interventions:  Provision  of  subsidy  to  farmers  on  the  selected  improved  farm  machineries & equipment’s used for various agricultural operat
[0.9209] Farm Machineries – Distribution of farm equipment and tools like conoweeder,  manual sprayer, drum seeder, seed drill, zero till multi­crop planter, p
[0.9280] Objectives: The prime objective would be to promote agriculture mechanization  through extending procurement subsidy & establishment of Farm Machinery


In [17]:
milvus_retriever = milvus_store.as_retriever(search_type="similarity", search_kwargs={"k": 3})
milvus_retriever.invoke(QUERY)


[Document(metadata={'pk': 25, 'source': 'data/sampled_schemes.pdf', 'source_kind': 'Pdf', 'page': 2, 'total_pages': 9}, page_content='Interventions:\xa0 Provision\xa0 of\xa0 subsidy\xa0 to\xa0 farmers\xa0 on\xa0 the\xa0 selected\xa0 improved\xa0 farm\xa0 machineries\xa0&\xa0equipment’s\xa0used\xa0for\xa0various\xa0agricultural\xa0operations\xa0(both\xa0during\xa0 crop\xa0stand\xa0&\xa0post\xadharvest),\xa0conducting\xa0training\xa0&\xa0demonstrations\xa0on\xa0their\xa0use\xa0and\xa0 maintenance.\xa0 Besides,\xa0 assistance\xa0 would\xa0 be'),
 Document(metadata={'pk': 8, 'source': 'data/sampled_schemes.pdf', 'source_kind': 'Pdf', 'page': 0, 'total_pages': 9}, page_content='Farm\xa0Machineries\xa0–\xa0Distribution\xa0of\xa0farm\xa0equipment\xa0and\xa0tools\xa0like\xa0conoweeder,\xa0 manual\xa0sprayer,\xa0drum\xa0seeder,\xa0seed\xa0drill,\xa0zero\xa0till multi\xadcrop\xa0planter,\xa0power\xa0 weeder,\xa0paddy\xa0thresher,laser\xa0land\xa0leveller;\xa0all\xa0at\xa0subsidized\xa0costs.\xa0

**Scaling, in brief:** of the four, Milvus is the one architected from the ground up for
distributed, billion-scale vector search — storage and compute are separate layers, and node
roles (proxy, query, data, index) scale independently of each other. Milvus Lite mirrors the same
`connection_args` / API surface for local dev, so the code above is genuinely the same code you'd
run against a production Milvus cluster or Zilliz Cloud, just pointed at a different URI.


## Retrieval comparison — same data, same query, four engines

All four stores above indexed the *exact same* `chunks` with the *exact same* `embeddings` —
unlike `l22_chunking_strategies.ipynb`, where different chunking changed what got retrieved,
here the point is the opposite: LangChain's common `VectorStore` interface makes the backend
swappable. Same data, same query, four different engines underneath, functionally interchangeable
top-k results.


In [18]:
stores = {
    "FAISS": faiss_store,
    "Chroma": chroma_store,
    "Qdrant": qdrant_store,
    "Milvus": milvus_store,
}

for name, store in stores.items():
    print(f"=== {name} ===")
    for doc, score in store.similarity_search_with_score(QUERY, k=3):
        print(f"[{score:.4f}] {doc.page_content[:120].replace(chr(10), ' ')}")
    print()


=== FAISS ===
[0.8007] Interventions:  Provision  of  subsidy  to  farmers  on  the  selected  improved  farm  machineries & equipment’s used f
[0.9209] Farm Machineries – Distribution of farm equipment and tools like conoweeder,  manual sprayer, drum seeder, seed drill, z
[0.9280] Objectives: The prime objective would be to promote agriculture mechanization  through extending procurement subsidy & e

=== Chroma ===
[0.8007] Interventions:  Provision  of  subsidy  to  farmers  on  the  selected  improved  farm  machineries & equipment’s used f
[0.9209] Farm Machineries – Distribution of farm equipment and tools like conoweeder,  manual sprayer, drum seeder, seed drill, z
[0.9280] Objectives: The prime objective would be to promote agriculture mechanization  through extending procurement subsidy & e

=== Qdrant ===
[0.5998] Interventions:  Provision  of  subsidy  to  farmers  on  the  selected  improved  farm  machineries & equipment’s used f
[0.5396] Farm Machineries – Distribution of 

All four agree on the same top-3 chunks, in the same order — that's the real proof the backend
is swappable. But look at the raw scores: FAISS, Chroma, and Milvus all print the *same*
increasing sequence (lower = better — squared-L2 distance, whether by default like Chroma or
explicitly like Milvus above), while Qdrant prints a different, *decreasing* sequence
(higher = better — cosine similarity, because that's what was explicitly configured for it
above). Ranking agrees across all four; raw score values and their direction do not.


## Takeaways

- All four vector stores implement the same LangChain `VectorStore` interface
  (`add_documents`, `similarity_search_with_score`, `as_retriever`) — the retrieval code barely
  changes between sections; only construction and connection details differ.
- "Local" doesn't mean "not production-capable" — Qdrant and Milvus's local modes are dev/test
  conveniences of databases whose real architecture is built for distributed scale; FAISS and
  Chroma's local modes largely *are* their production story, just with more or less tooling
  around them.
- Score direction isn't standardized — FAISS, Chroma (left at its default), and Milvus
  (explicitly configured) all use L2 distance (lower = better) above, while only Qdrant was
  explicitly configured with cosine similarity (higher = better). That split is a config choice,
  not an inherent property of any store — always check which metric a store is actually
  configured with before comparing raw scores across stores.
- This is the natural upgrade from `l21_rag_pipeline_from_scratch.ipynb`'s from-scratch cosine
  similarity: same retrieval concept, now with persistence, real ANN indexes, and (for Qdrant and
  Milvus) a real path to distributed production scale.
